<a href="https://www.kaggle.com/code/pawankumargunjan/name-that-llm-tfidf-sgdclassifier?scriptVersionId=373348084" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Import packages

In [1]:
import os
import numpy as np
import pandas as pd
import pathlib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GroupKFold, cross_val_score

## Read Data

In [2]:
INPUT_DIR = pathlib.Path("/kaggle/input")

COMPETITIONS_DIR = sorted([
    p for p in (INPUT_DIR / "competitions").iterdir()
    if p.is_dir()
])

DATASET_DIR = sorted([
    p for p in (INPUT_DIR / "datasets").iterdir()
    if p.is_dir()
])

# MODEL_DIR = sorted([
#     p for p in (INPUT_DIR / "models").iterdir()
#     if p.is_dir()
# ])


print("Competitions directories:", len(COMPETITIONS_DIR))
for path in COMPETITIONS_DIR:
    for file_path in path.rglob("*"):
        if file_path.is_file():
            path_str = str(file_path)
            print(path_str)
    
print("\nDataset directories:", len(DATASET_DIR))
for path in DATASET_DIR:
    print(path)

# print("\nModel directories:", len(MODEL_DIR))
# for path in MODEL_DIR:
#     print(path)

Competitions directories: 1
/kaggle/input/competitions/name-that-llm/sample_submission.parquet
/kaggle/input/competitions/name-that-llm/train.csv
/kaggle/input/competitions/name-that-llm/test.csv

Dataset directories: 1
/kaggle/input/datasets/pawankumargunjan


## Read the datasets

In [3]:
# competitions training data
TRAIN_PATH = (
    INPUT_DIR / "competitions/name-that-llm/train.csv"
)

# Original competition dataset
train_df = pd.read_csv(TRAIN_PATH)
print("Original training shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())
print("Classes:", train_df["model"].nunique())

VALID_LABELS = set(train_df["model"].dropna().unique())
train_df.head()

Original training shape: (2699, 4)
Columns: ['id', 'prompt_id', 'text', 'model']
Classes: 18


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [4]:
VALID_MODEL_NAMES = set(train_df['model'].unique())

# # Create label mappings if your 'model' column contains strings
# label2id = {model: i for i, model in enumerate(VALID_MODEL_NAMES)}
# id2label = {i: model for i, model in enumerate(VALID_MODEL_NAMES)}

# id2label
print(VALID_MODEL_NAMES)

{'claude-opus-4-8', 'gemini-3.6-flash', 'claude-sonnet-5', 'gpt-5.6-terra', 'claude-opus-5', 'gemini-3.1-pro-preview', 'gpt-5.5', 'gpt-5.6-sol', 'gpt-5.6-luna', 'gemini-3.7-flash', 'kimi-k3', 'qwen3.8-max', 'gpt-6-astra', 'glm-5.3-flash', 'claude-opus-4-6', 'claude-fable-5', 'glm-5.3', 'gemini-3.8-flash'}


## Combined with Extra Data

In [5]:
# competitions training data
FINAL_DATASET_PATH = (
    INPUT_DIR / "datasets/pawankumargunjan/name-that-llm-consolidated/train_english.csv"
)
consolidated_df = pd.read_csv(FINAL_DATASET_PATH)

print("Dataset shape:", consolidated_df.shape)
print("Columns:", consolidated_df.columns.tolist())

display(consolidated_df.head())

Dataset shape: (203670, 4)
Columns: ['text', 'model', 'text_length', 'language']


,text,model,text_length,language
0,A single photon leaves a distant star and trav...,gpt-6-astra,1031,en
1,"There is a hum where a heartbeat used to be, a...",claude-sonnet-5,425,en
2,"Deep beneath the churning crust, within the cr...",gemini-3.7-flash,562,en
3,Observing a thousand white swans merely confir...,glm-5.3-flash,414,en
4,If the universe unfolds according to fixed phy...,claude-sonnet-5,1643,en


In [6]:
consolidated_df.describe()

,text_length
count,203670.000000
mean,2745.640747
std,4290.062523
min,50.000000
25%,548.000000
50%,1392.000000
75%,3812.000000
max,631611.000000


In [7]:
consolidated_df['model'].value_counts()

model
gpt-5.6-luna              81464
claude-opus-4-8           23914
glm-5.3-flash             18168
claude-opus-4-6           14037
gpt-5.6-sol                7761
kimi-k3                    7462
claude-opus-5              6217
claude-sonnet-5            6181
gpt-5.6-terra              6142
gpt-5.5                    6110
claude-fable-5             4868
gemini-3.1-pro-preview     3945
qwen3.8-max                3254
gpt-6-astra                3232
glm-5.3                    2996
gemini-3.7-flash           2676
gemini-3.6-flash           2648
gemini-3.8-flash           2595
Name: count, dtype: int64

In [8]:
unique_models = sorted(consolidated_df['model'].unique())

# Create label mappings if your 'model' column contains strings
#label2id = {model: i for i, model in enumerate(unique_models)}
id2label = {i: model for i, model in enumerate(unique_models)}

id2label

{0: 'claude-fable-5',
 1: 'claude-opus-4-6',
 2: 'claude-opus-4-8',
 3: 'claude-opus-5',
 4: 'claude-sonnet-5',
 5: 'gemini-3.1-pro-preview',
 6: 'gemini-3.6-flash',
 7: 'gemini-3.7-flash',
 8: 'gemini-3.8-flash',
 9: 'glm-5.3',
 10: 'glm-5.3-flash',
 11: 'gpt-5.5',
 12: 'gpt-5.6-luna',
 13: 'gpt-5.6-sol',
 14: 'gpt-5.6-terra',
 15: 'gpt-6-astra',
 16: 'kimi-k3',
 17: 'qwen3.8-max'}

## Feature Extraction Pipeline

In [9]:
word_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=20000,
    sublinear_tf=True,
    dtype=np.float32
)

char_tfidf = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=35000,
    sublinear_tf=True,
    dtype=np.float32
)

feature_union = FeatureUnion([
    ("word_tfidf", word_tfidf),
    ("char_tfidf", char_tfidf)
])

## Building the Pipeline

In [10]:
sgd_pipeline = Pipeline([
    ("features", feature_union),
    ("clf", SGDClassifier(
        loss="log_loss",
        max_iter=1000,
        random_state=42,
        n_jobs=1
    ))
])

## K-Fold Cross validations

In [11]:
# # 5-fold CV that keeps each prompt in a single fold (test prompts are unseen), scored like the leaderboard
# cv_scores = cross_val_score(
#     sgd_pipeline,
#     consolidated_df['text'].fillna(''),
#     consolidated_df['model'],
#     groups=consolidated_df['model'],
#     cv=GroupKFold(n_splits=5),
#     scoring='balanced_accuracy',
#     n_jobs=-1,
# )
# print(f'5-Fold Grouped CV Balanced Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

## Fitting on the Full Training Set

In [12]:
sgd_pipeline.fit(
    consolidated_df['text'].fillna(''),
    consolidated_df['model']
)

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(dtype=<class 'numpy.float32'>,
                                                                 max_features=20000,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 sublinear_tf=True)),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char',
                                                                 dtype=<class 'numpy.float32'>,
                                                                 max_features=35000,
                                                                 ngram_range=(3,
                                                                              5),
                                                                 sublinear_tf=True))])),
                ('clf',
                 SGDClassifier(loss='log_loss', n_jobs=1, random_state=42))])

## Submission

In [13]:
# Dynamically find the file paths anywhere inside /kaggle/input
test_path = next(INPUT_DIR.rglob("test.csv"))
#submission_path = next(input_dir.rglob("sample_submission.parquet"))

# Load the datasets using the dynamic paths
df_test = pd.read_csv(test_path)
#df_submission = pd.read_parquet(submission_path)

# Print shapes and inspect
print(f"Test: {df_test.shape}") #nSubmission: {df_submission.shape}
df_test.head()

Test: (3, 2)


,id,text
0,00353fb1c541,"The more efficiently we use a resource, the mo..."
1,005f5a9a6139,"The Stern-Gerlach experiment, performed in 192..."
2,0062a53d5487,"Hundreds of millions of years ago, fluctuating..."


In [14]:
# Run inference on the actual competition test set
print("Running predictions on competition test data...")
test_preds = sgd_pipeline.predict(df_test['text'].fillna(''))

# Match test IDs with submission IDs
submission_df = pd.DataFrame({
    "id": df_test["id"],
    "model": test_preds
})


submission_df.to_parquet('submission.parquet', index=False)
print(f'Saved submission.parquet with {len(submission_df)} rows.')
submission_df.head(10)

Running predictions on competition test data...
Saved submission.parquet with 3 rows.


,id,model
0,00353fb1c541,gpt-5.6-luna
1,005f5a9a6139,claude-opus-4-6
2,0062a53d5487,claude-opus-4-6
